# 05. Xe II: свой расчёт рассеяния в модели Ванга — проверка, метастабили, каскады

Три вещи, которых нет в ноутбуке 02 (там σ берутся готовыми у Ванга):

1. **Проверка σ Ванга.** Та же мишень (`models.xe2_wang`, 68 состояний, нумерация уровней —
   таблица Ванга), свой расчёт DBSR. Сравниваются σ и ⟨σv⟩ для **всех** пар, которые есть у
   Ванга (gs→6s/5d/6p, 6s→6p, 5d→6p). Это воспроизведение расчёта, а не другая физическая
   модель: совпадение проверяет, что Ванг и мы посчитали одно и то же (сетка, число
   парциальных волн, пороги). Зависимость от модели мишени — ноутбук 06.
2. **Переходы между метастабилями** (и с ²P₁/₂): у Ванга их нет. Метастабиль здесь — уровень
   ниже 6p, у которого время жизни по E1 (из этого же расчёта) длиннее `TAU_META` или
   E1-распада нет вовсе. M1 и E2 не считаются (dbsr_dmat3 падает на M1), поэтому настоящие
   времена жизни короче — проверьте по литературе, если важно.
3. **Каскады в верхние уровни выбранных линий.** Для каждого уровня h выше u — вероятность
   P(h→u), что h, распадаясь, пройдёт через u (все цепочки, ветвления DBSR E1). Константа
   каскадного заселения из уровня i: k_casc(i→u) = Σ_h k(i→h)·P(h→u). Отношение
   k_casc/k(i→u) — атомные данные (как σ_app у Fursa); как каскады войдут в заселённость, решает CRM.
   В модели Ванга выше 6p есть только 5d и 7s (до 18.5 эВ); 6d, 7p, 4f — ноутбук 06.

Все σ здесь — DBSR (этот расчёт); ветвления — DBSR E1 (этот расчёт).

**Время**: J = 0 (2 волны) на 4 ядрах — см. `docs/crm/lines_final.md`; полный расчёт
(J ≤ 25) — сервер. Сходимость по J проверьте (`JMAX` 15/20/25): у оптически разрешённых
переходов (gs→6s, 5d) большие J вносят заметный вклад.

**Выход** (папка готового расчёта `runs/xe_plus_v2`, если он есть, иначе `runs/crm/xe2_wang_full`): `sigma_xe2_wang.xlsx` (σ из начальных уровней Ванга,
формат Ванга), `rates_xe2_wang.xlsx` (k всех возбуждений), `check_vs_wang.csv`,
`xe2_levels_tau.csv`, `xe2_sigma_metastable.xlsx`, `xe2_k_metastable.xlsx`,
`xe2_cascade_P.csv`, `xe2_k_cascade.xlsx` (+ .csv).

In [ ]:
# ---------- параметры (поменяйте пути под себя)
from pathlib import Path
import sys
ROOT    = Path("/data/files_Otdel120/pydbsr_calc")    # папка установки (большой диск)
REPO    = ROOT / "pydbsr_src"                         # распакованный архив pydbsr (или клон репозитория)
CRM     = REPO / "examples" / "crm"                   # nbtools.py, run_model.py, models.py
NIST    = REPO / "docs" / "crm"                       # выгрузки NIST ASD (nist_*.csv)
DATA    = ROOT                                        # ваши файлы
XLSX    = DATA / "CrossSectionsIon_3.xlsx"            # Wang et al (2019), сечения e + Xe+
RUNS    = ROOT / "runs" / "crm"                       # результаты расчётов (большой диск)
XE2_RUN = ROOT / "runs" / "xe_plus_v2"                # готовый расчёт e + Xe+ (модель Ванга, J <= 25)
SCRATCH, SCRATCH_GB = Path.home() / "pydbsr_tmp", 5   # временные файлы dbsr_mat (быстрый диск)
CORES, MEM, HD_THREADS = 32, 100, 8                   # ядра, ГБ памяти, потоки диагонализации
TE = [0.3, 0.5, 0.7, 1.0, 1.5, 2.0, 3.0, 5.0, 7.0, 10.0, 15.0, 20.0]   # сетка Te (эВ) для констант скоростей
sys.path.insert(0, str(CRM))
import numpy as np, matplotlib.pyplot as plt
import pydbsr as db
from pydbsr import rates, reference
import nbtools as nb
%matplotlib inline

In [ ]:
import shutil
JMAX, EMAX = 25, 60.0        # J: проверьте сходимость; EMAX (эВ): σ выше — хвост 1/E (как в 02)
ref = reference.read_xlsx(XLSX)
WANG_INITIAL = sorted({i for i, j in ref.sigma})         # 1, 2, 4, 5, ... 34
W = RUNS / "xe2_wang_full"
if list((XE2_RUN / "scat").glob("h.[0-9][0-9][0-9]")):
    # готовый расчёт xe_plus_v2 (examples/xe_plus_wang2019.ipynb) — та же модель, сетка энергий
    # 0.0136 эВ до 60 эВ: рассеяние не пересчитывается, считаются только A, σ и k(Te)
    W = XE2_RUN
    om = sorted((W / "compare").glob(f"omega_J{JMAX:g}_E*.npz"))
    if om and not (W / f"omega_J{JMAX:g}.npz").exists():
        (W / f"omega_J{JMAX:g}.npz").symlink_to(om[0])
    print("готовый расчёт:", W, "| парциальных волн:", len(list((W / "scat").glob("h.[0-9][0-9][0-9]"))),
          "| Ω:", (W / f"omega_J{JMAX:g}.npz").resolve())
elif (RUNS / "xe2_wang" / "target").exists() and not (W / "target").exists():
    shutil.copytree(RUNS / "xe2_wang" / "target", W / "target")      # мишень из ноутбука 01
job = nb.Job(W, "xe2_wang", levels=XLSX, stage="all", jmax=JMAX, cores=CORES, mem=MEM, hd_threads=HD_THREADS,
             scratch=SCRATCH, scratch_gb=SCRATCH_GB, emax=EMAX, de=0.0136, multipoles="E1",
             sigma_initial=",".join(map(str, WANG_INITIAL)), te=",".join(map(str, TE)))
job.start()

In [ ]:
job.status()          # ход расчёта (запускайте когда хотите)

In [ ]:
cs = db.CollisionStrengths.load(W / f"omega_J{JMAX:g}.npz", partial_waves=False)
tg = db.Target.load(W / "target")
name = {s.nist_no: s.name for s in tg.states if s.nist_no and s.name in cs.names}   # номер Ванга -> состояние
Ecm = {lv.no: lv.energy_cm for lv in ref.levels}
def dbsr_sigma(i, j):
    E, s = cs.incident_energy(name[i]), cs.sigma(name[i], name[j])
    m = np.isfinite(s) & (E > 0)
    return E[m], s[m]
def thr(i, j):
    return (Ecm[j] - Ecm[i]) / nb.CM_PER_EV

TEa = np.array(TE)
def k_of(E, s, thr_ev):
    """Maxwellian <sigma v> (cm3/s) on the TE grid; tail above the last energy: 1/E."""
    return rates.rates_on_grid(E, s, TEa, thr_ev)
def show_k(title, k):
    print(f"{title:28s} " + "  ".join(f"{v:9.2e}" for v in k))
print(" " * 28 + "  ".join(f"Te={t:<6g}" for t in TEa))

## 1. σ Ванга против своего расчёта

Для каждой пары: k_DBSR/k_Ванг при каждом Te (обе константы — из σ одной и той же функцией
`rates.rates_on_grid`, хвост 1/E). Сводка — медиана и доля пар в пределах ×1.5 и ×2.

In [ ]:
import csv
check = []
for (i, j), (Ew, sw) in sorted(ref.sigma.items()):
    if i in name and j in name:
        kw = k_of(Ew, sw, thr(i, j))
        kd = k_of(*dbsr_sigma(i, j), thr(i, j))
        check.append(dict(i=i, j=j, label=f"{ref.label(i)} -> {ref.label(j)}",
                          **{f"ratio_Te{t:g}": d / w for t, d, w in zip(TEa, kd, kw)}))
for t in TEa:
    q = np.array([c[f"ratio_Te{t:g}"] for c in check])
    q = q[np.isfinite(q) & (q > 0)]
    print(f"Te = {t:5g} эВ: {len(q)} пар, медиана DBSR/Ванг = {np.median(q):.2f}, "
          f"в пределах ×1.5: {np.mean((q > 1 / 1.5) & (q < 1.5)):.0%}, ×2: {np.mean((q > 0.5) & (q < 2)):.0%}")
with open(W / "check_vs_wang.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(check[0])); w.writeheader(); w.writerows(check)
print("->", W / "check_vs_wang.csv")
print("\nверхние уровни выбранных линий:")
for c in check:
    if c["j"] in nb.XE2_UPPERS:
        print(f"{c['i']:3d} -> {c['j']:3d}  " + "  ".join(f"{c[f'ratio_Te{t:g}']:5.2f}" for t in TEa))

In [ ]:
fig, axs = plt.subplots(2, len(nb.XE2_LINES), figsize=(4 * len(nb.XE2_LINES), 6.5), squeeze=False)
for col, x in enumerate(nb.XE2_LINES):
    u = x["upper"]
    for row, i in enumerate((1, 2)):
        ax = axs[row][col]
        if (i, u) in ref.sigma:
            E, s = ref.sigma[(i, u)]; ax.plot(E, s / 1e-16, "k", lw=0.8, label="Ванг 2019")
        E, s = dbsr_sigma(i, u); ax.plot(E, s / 1e-16, "r", lw=0.6, label=f"DBSR, J ≤ {JMAX}")
        ax.set_xscale("log"); ax.set_title(f"{i} → {u} ({x['wl']} нм)", fontsize=9)
axs[0][0].legend(fontsize=7); fig.supxlabel("E, эВ"); fig.supylabel("σ, 10⁻¹⁶ см²"); fig.tight_layout(); plt.show()

## 2. Времена жизни (E1) и метастабили

`TAU_META` — порог: уровни ниже 6p с τ(E1) длиннее него (или без E1-распада) считаются
метастабилями. Уровень 2 (²P₁/₂) распадается только M1 — он всегда в списке.

In [ ]:
TAU_META = 1e-6            # с
br, tot, A = nb.branching_from_csv(W / "transitions_E1.csv")
first6p = min(n for n in name if ")6p" in ref.label(n))
low = [n for n in sorted(name) if Ecm[n] < Ecm[first6p]]
META = [n for n in low if n > 1 and (n not in tot or 1 / tot[n] > TAU_META)]
rows = []
for n in sorted(name):
    tau = 1 / tot[n] if n in tot else float("inf")
    rows.append(dict(no=n, label=ref.label(n), E_eV=Ecm[n] / nb.CM_PER_EV, tau_E1_s=tau, metastable=n in META))
    if n in low:
        print(f"{n:3d} {ref.label(n):34s} {Ecm[n] / nb.CM_PER_EV:7.3f} эВ   τ(E1) = {tau:9.3g} с" + ("   метастабиль" if n in META else ""))
with open(W / "xe2_levels_tau.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0])); w.writeheader(); w.writerows(rows)
print("метастабили:", META, " (начальные уровни у Ванга:", WANG_INITIAL, ")")

## 3. σ и ⟨σv⟩ между основным, ²P₁/₂ и метастабилями

Набор M — 1, 2, метастабили и начальные уровни Ванга ниже 6p (у Ванга это и 6s/5d с
τ ≈ 0.1–0.2 мкс, которые в плазме тоже заметно заселены). Только возбуждение i → j (E_i < E_j). Обратный процесс — детальное равновесие:
`rates.deexcitation(k, g_i, g_j, ΔE, Te)`, g = 2J+1 (столбец J в «NIST Level Table»).
Плюс k(i → u) из этих уровней в верхние уровни выбранных линий (у Ванга есть не все).

In [ ]:
M = sorted({1, 2} | set(META) | {i for i in WANG_INITIAL if Ecm[i] < Ecm[first6p]})   # + начальные уровни Ванга ниже 6p
sig_m, lab_m, k_m = {}, {}, {}
for a_, i in enumerate(M):
    for j in M[a_ + 1:] + [u for u in nb.XE2_UPPERS]:
        if Ecm[j] <= Ecm[i] or (i, j) in sig_m:
            continue
        E, s = dbsr_sigma(i, j)
        sig_m[(i, j)] = (E, s)
        lab_m[(i, j)] = f"{i}->{j}" + ("" if (i, j) in ref.sigma or j not in nb.XE2_UPPERS else " (нет у Ванга)")
        k_m[(i, j)] = k_of(E, s, thr(i, j))
reference.write_xlsx(W / "xe2_sigma_metastable.xlsx", ref.levels, sig_m, sheet_of=lambda k: f"from {k[0]}", labels=lab_m)
reference.write_rates_xlsx(W / "xe2_k_metastable.xlsx", TEa, k_m, lab_m,
                           note=f"<sigma v> (cm3/s), excitation i->j, levels: Wang 2019 numbers; sigma: DBSR, model xe2_wang, J<={JMAX}")
for key in sorted(k_m):
    show_k(lab_m[key], k_m[key])

## 3б. Проверка через силы линий (gf): какому набору σ из метастабилей верить

Возбуждение m → u (m — уровень 6s/5d, u — 6p) и излучение u → m — один и тот же переход.
Для разрешённого перехода σ(m→u) примерно пропорционально его силе линии (gf ∝ A):
при больших энергиях строго, у порога — приближённо. Наш расчёт даёт gf всех таких
переходов, NIST — для части из них. Если наш A(u→m) больше NIST в X раз, то и наше
σ(m→u), вероятно, завышено примерно в X раз.

Столбцы:
* `наш/NIST` — наш A против NIST (нет в NIST — пусто);
* `наш/изм.` — наш A против измеренного по вашим спектрам: (доля ветви по DBSR) / (измеренная
  доля), медиана по всем спектрам `docs/crm/branching/` (только верхние уровни ваших линий
  и видимые ветви). Где есть оба, верим спектру (для 680.57 нм спектр противоречит NIST);
* `vel/len` — согласие двух форм нашего расчёта (далеко от 1 — линии не верим);
* `k_наш/k_Ванг` — при `TE_CHECK` из раздела 1;
* `исправл./Ванг` — k_наш × (A_истинное/A_наш) / k_Ванг: если близко к 1, расхождение с Вангом
  объясняется силой линии в нашей модели, и правее Ванг; если далеко от 1 — дело не в A.

In [ ]:
TE_CHECK = 1.0                        # эВ
lines_nist = nb.nist_lines(NIST / "nist_XeII.csv")
nos = sorted(Ecm, key=Ecm.get)        # номера уровней Ванга по возрастанию энергии
E_sorted = [Ecm[n] for n in nos]
A_nist = {}
for x in lines_nist:
    if not x["A"] or x["Ei"] is None or x["Ek"] is None:
        continue
    lo = nb.level_number(E_sorted, x["Ei"], tol=1.0)
    up = nb.level_number(E_sorted, x["Ek"], tol=1.0)
    if lo and up:
        A_nist[(nos[up - 1], nos[lo - 1])] = x["A"]
A_our, gauge = {}, {}
for r in nb.read_csv(W / "transitions_E1.csv"):
    if r["upper_no"] and r["lower_no"]:
        key = (int(r["upper_no"]), int(r["lower_no"]))
        A_our[key] = nb.fnum(r["A_exp"])
        gauge[key] = nb.fnum(r["gauge_ratio"])
# измерено по спектрам (examples/crm/branching_from_spectrum.py): доля ветви среди видимых × (1 − доля
# невидимых по DBSR, строка «ИТОГ») = измеренное ветвление; / ветвление DBSR ≈ A_истинное / A_наш
meas = {}
for fcsv in sorted((NIST / "branching").glob("branching_*.csv")):
    rs = nb.read_csv(fcsv)
    unseen = {r["upper"]: nb.fnum(r["unseen_dbsr"]) for r in rs if r["branch_nm"] == "ИТОГ"}
    for r in rs:
        share, bd = nb.fnum(r["share_of_seen"]), nb.fnum(r["BR_dbsr"])
        if share and bd and r["lower"].strip().isdigit() and unseen.get(r["upper"]) is not None:
            meas.setdefault((int(r["upper"]), int(r["lower"])), []).append(share * (1 - unseen[r["upper"]]) / bd)
meas = {k: float(np.median(v)) for k, v in meas.items()}
ck = {(c["i"], c["j"]): c for c in check}
rows = []
print(f"{'m->u':>8s}  {'λ, нм':>8s}  {'A_наш':>9s}  {'A_NIST':>9s}  {'наш/NIST':>8s}  {'наш/изм.':>8s}  {'vel/len':>7s}"
      f"  {'k_наш/k_Ванг':>12s}  {'исправл./Ванг':>13s}")
for u in nb.XE2_UPPERS:
    for m in sorted(M):
        if (u, m) not in A_our or Ecm[m] >= Ecm[u]:
            continue
        a, an = A_our[(u, m)], A_nist.get((u, m))
        mr = meas.get((u, m))                          # A_истинное / A_наш по спектру
        kr = ck.get((m, u), {}).get(f"ratio_Te{TE_CHECK:g}")
        true_over_our = mr if mr else (an / a if (an and a) else None)   # спектр важнее NIST
        fix = kr * true_over_our if (kr and true_over_our) else None
        wl = 1e7 / (Ecm[u] - Ecm[m]) / 1.000277
        rows.append(dict(m=m, u=u, wl_nm=wl, A_our=a, A_NIST=an, our_NIST=a / an if an else None,
                         our_measured=1 / mr if mr else None,
                         vel_len=gauge.get((u, m)), k_our_Wang=kr, corrected_Wang=fix))
        f = lambda v, fmt: format(v, fmt) if v is not None else "—"
        print(f"{m:3d}->{u:<3d}  {wl:8.3f}  {a:9.2e}  {f(an, '9.2e'):>9s}  {f(a / an if an else None, '8.2f'):>8s}"
              f"  {f(1 / mr if mr else None, '8.2f'):>8s}"
              f"  {f(gauge.get((u, m)), '7.2f'):>7s}  {f(kr, '12.2f'):>12s}  {f(fix, '13.2f'):>13s}")
with open(W / "xe2_gf_check.csv", "w", newline="") as fh:
    wr = csv.DictWriter(fh, fieldnames=list(rows[0])); wr.writeheader(); wr.writerows(rows)
print("->", W / "xe2_gf_check.csv")

## 4. Каскады в верхние уровни выбранных линий

P(h→u) — доля распадов h, проходящих через u (ветвления DBSR E1, все цепочки). Печатаются
уровни с P > `P_MIN`. k_casc(i→u) = Σ_h k(i→h)·P(h→u); отношение к прямому k(i→u).

In [ ]:
P_MIN = 1e-3
P = nb.cascade_prob(br)
prow, k_c, lab_c = [], {}, {}
kcache = {}
def k_pair(i, j):
    if (i, j) not in kcache:
        kcache[(i, j)] = k_of(*dbsr_sigma(i, j), thr(i, j))
    return kcache[(i, j)]
for x in nb.XE2_LINES:
    u = x["upper"]
    feed = sorted(((h, P[h][u]) for h in P if P[h].get(u, 0) > P_MIN and h in name), key=lambda t: -t[1])
    print(f"\n{x['wl']} нм, верхний уровень {u} {ref.label(u)}: питающих уровней {len(feed)}")
    for h, p in feed:
        prow.append(dict(upper=u, line_nm=x["wl"], h=h, h_label=ref.label(h), P=p))
        print(f"   {h:3d} {ref.label(h):34s} P = {p:.3f}")
    for i in M:
        if Ecm[i] >= Ecm[u]:
            continue
        kc = sum(k_pair(i, h) * p for h, p in feed if Ecm[h] > Ecm[i])
        kd = k_pair(i, u)
        k_c[(i, u, "casc")] = kc; lab_c[(i, u, "casc")] = f"{i}->{u} cascade"
        k_c[(i, u, "ratio")] = kc / kd; lab_c[(i, u, "ratio")] = f"{i}->{u} cascade/direct"
        if i in (1, 2):
            show_k(f"  из {i}: k_casc/k_прям", kc / kd)
with open(W / "xe2_cascade_P.csv", "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(prow[0])); w.writeheader(); w.writerows(prow)
reference.write_rates_xlsx(W / "xe2_k_cascade.xlsx", TEa, k_c, lab_c,
                           note="cascade: sum_h <sigma v>(i->h) P(h->u), cm3/s; ratio = cascade/direct; DBSR, model xe2_wang")
print("->", W / "xe2_cascade_P.csv", W / "xe2_k_cascade.xlsx")

## 5. Набор k с поправкой по спектру (третий набор для CRM)

Для переходов m → u, у которых настоящая сила линии известна (ваш спектр — в приоритете,
иначе NIST, раздел 3б), наше k(m→u) умножается на A_истинное / A_наш. Это поправка
дипольной части сечения: точна при больших энергиях, у порога — приближённо (там вклад
обмена и резонансов не масштабируется). Для остальных пар k не меняются — берите их из
`rates_xe2_wang.xlsx`.

В CRM: три набора k(метастабиль → 6p) — Ванг, наш, наш с поправкой. Разброс ne по ним —
неопределённость от атомных данных; поправленный набор лучше всего согласован с вашим спектром.

In [ ]:
k_cor, lab_cor, cor_rows = {}, {}, []
for (u, m), a in sorted(A_our.items()):
    if m not in M or u not in nb.XE2_UPPERS or Ecm[m] >= Ecm[u] or not a:
        continue
    f_true = meas.get((u, m)) or (A_nist[(u, m)] / a if (u, m) in A_nist else None)
    if not f_true:
        continue
    src = "спектр" if (u, m) in meas else "NIST"
    k_cor[(m, u)] = f_true * k_pair(m, u)
    lab_cor[(m, u)] = f"{m}->{u}"
    cor_rows.append(dict(m=m, u=u, factor=f_true, source=src))
    show_k(f"{m}->{u} ×{f_true:.2f} ({src})", k_cor[(m, u)])
reference.write_rates_xlsx(W / "xe2_k_corrected.xlsx", TEa, k_cor, lab_cor,
                           note="<sigma v> (cm3/s), m->u: DBSR (xe_plus_v2) x A_true/A_DBSR (A_true: branching measured "
                                "from the spectra, else NIST); other pairs: rates_xe2_wang.xlsx")
with open(W / "xe2_k_corrected_factors.csv", "w", newline="") as fh:
    wr = csv.DictWriter(fh, fieldnames=list(cor_rows[0])); wr.writeheader(); wr.writerows(cor_rows)
print("->", W / "xe2_k_corrected.xlsx", W / "xe2_k_corrected_factors.csv")

## 6. Поиск ne-пар, устойчивых к выбору атомных данных

Для каждой видимой линии Xe II (спектр A3: не в бленде, не насыщена хотя бы в одном листе,
S/N ≥ `SN_MIN`; таблица `docs/crm/criteria_A3.csv`) с верхним уровнем u из 6p:

* «рычаг метастабилей» L(u) = Σ_m w_m k(m→u) / k(1→u), m — `LEVER_SET` (метастабили и
  долгоживущий уровень 4, у которых σ есть и у Ванга, и у нас); w_m — относительные веса
  (по умолчанию равные; если ваша CRM даёт заселённости, поставьте их);
* «наклон» s(u) = d ln k(1→u) / d ln Te между `TE_PAIR` и 2·`TE_PAIR`.

Хорошая ne-пара (a, b): (1) **чувствительна** — L(a)/L(b) далеко от 1 в обоих наборах;
(2) **нечувствительна к Te** — наклоны близки; (3) **устойчива** — наше L(a)/L(b) и Ванговское
L(a)/L(b) близки, т. е. ответ не зависит от того, чьи сечения взять. Столбцы: `чувств.` =
min(|ln L_a/L_b|) по двум наборам, `расхожд.` = |ln (L_a/L_b)_наш − ln (L_a/L_b)_Ванг|,
`Δнаклон`, `Δλ` (нм; чем меньше, тем меньше нужна калибровка), доля «6s» в рычаге — сколько
дают уровни 6s (там наборы согласны, раздел 3б).

In [ ]:
TE_PAIR = 1.0                  # эВ
SN_MIN = 50
LEVER_SET = [m for m in (2, 4, 7, 10, 12, 22) if m in name]
LEVER_W = {m: 1.0 for m in LEVER_SET}           # веса метастабилей (заселённости относительно основного)
SIX_S = {m for m in LEVER_SET if ")6s" in ref.label(m)}
TE2 = np.array([TE_PAIR, 2 * TE_PAIR])

def k_both(i, u):
    """(k_наш, k_Ванг) на TE2 или None, если у Ванга нет σ."""
    if (i, u) not in ref.sigma:
        return None
    Ew, sw = ref.sigma[(i, u)]
    kd = rates.rates_on_grid(*dbsr_sigma(i, u), TE2, thr(i, u))
    kw = rates.rates_on_grid(Ew, sw, TE2, thr(i, u))
    return kd, kw

crit = [r for r in nb.read_csv(NIST / "criteria_A3.csv")
        if r["ion"] == "XeII" and r["upper_no"] and int(r["upper_no"]) in name and ")6p" in ref.label(int(r["upper_no"]))
        and not r["nist_within_1.5fwhm"]]
def best_sn(r):
    v = [nb.fnum(r[k]) for k in r if k.endswith("_S/N") and r[k]]
    return max(v) if v else 0.0
def n_unsat(r):
    return sum(1 for k in r if k.endswith("_sat") and r[k.replace("_sat", "_S/N")] and r[k] != "SAT")
crit = [r for r in crit if best_sn(r) >= SN_MIN and n_unsat(r) > 0]
lever = {}
for u in sorted({int(r["upper_no"]) for r in crit}):
    k1 = k_both(1, u)
    if k1 is None:
        continue
    terms = {m: k_both(m, u) for m in LEVER_SET if Ecm[m] < Ecm[u]}
    if any(t is None for t in terms.values()):
        continue
    LD = sum(LEVER_W[m] * terms[m][0][0] for m in terms) / k1[0][0]
    LW = sum(LEVER_W[m] * terms[m][1][0] for m in terms) / k1[1][0]
    share6s = sum(LEVER_W[m] * terms[m][0][0] for m in terms if m in SIX_S) / (LD * k1[0][0])
    sD = np.log(k1[0][1] / k1[0][0]) / np.log(2.0)
    sW = np.log(k1[1][1] / k1[1][0]) / np.log(2.0)
    lever[u] = dict(LD=LD, LW=LW, s=(sD + sW) / 2, sD=sD, sW=sW, six_s=share6s)
lines_c = [r for r in crit if int(r["upper_no"]) in lever]
print(f"линий-кандидатов: {len(lines_c)}, верхних уровней: {len(lever)}")
for u, v in sorted(lever.items()):
    print(f"  {u:3d} {ref.label(u):30s} L_наш = {v['LD']:9.2e}  L_Ванг = {v['LW']:9.2e}  наклон = {v['s']:5.2f}"
          f"  доля 6s = {v['six_s']:.2f}")
pairs = []
for ia, a in enumerate(lines_c):
    for b in lines_c[ia + 1:]:
        ua, ub = int(a["upper_no"]), int(b["upper_no"])
        if ua == ub:
            continue
        la, lb = lever[ua], lever[ub]
        rD, rW = np.log(la["LD"] / lb["LD"]), np.log(la["LW"] / lb["LW"])
        pairs.append(dict(line_a=float(a["wl_nm"]), up_a=ua, line_b=float(b["wl_nm"]), up_b=ub,
                          sens=min(abs(rD), abs(rW)) if rD * rW > 0 else 0.0, disagree=abs(rD - rW),
                          dslope=abs(la["s"] - lb["s"]), dwl=abs(float(a["wl_nm"]) - float(b["wl_nm"])),
                          six_s_a=la["six_s"], six_s_b=lb["six_s"],
                          sn_a=best_sn(a), sn_b=best_sn(b)))
good = {}
for p in pairs:                       # одна пара линий на пару уровней: ближайшая по λ
    if p["sens"] > np.log(2) and p["dslope"] < 0.5:
        key = frozenset((p["up_a"], p["up_b"]))
        if key not in good or p["dwl"] < good[key]["dwl"]:
            good[key] = p
good = sorted(good.values(), key=lambda p: (p["disagree"] / p["sens"], p["dwl"]))
print(f"\nпар линий: {len(pairs)}; пар уровней, чувствительных (L_a/L_b > 2 в обоих наборах) и с Δнаклон < 0.5:"
      f" {len(good)} (лучшие сверху: малое расхождение наборов при большой чувствительности)")
print(f"{'линия a':>9s} {'ур.':>4s} {'линия b':>9s} {'ур.':>4s} {'чувств.':>7s} {'расхожд.':>8s} {'Δнаклон':>7s}"
      f" {'Δλ':>6s} {'6s a/b':>9s}")
for p in good[:25]:
    print(f"{p['line_a']:9.3f} {p['up_a']:4d} {p['line_b']:9.3f} {p['up_b']:4d} {p['sens']:7.2f} {p['disagree']:8.2f}"
          f" {p['dslope']:7.2f} {p['dwl']:6.1f} {p['six_s_a']:4.2f}/{p['six_s_b']:4.2f}")
ref_pair = [p for p in pairs if {p["line_a"], p["line_b"]} == {561.667, 545.045}]
if ref_pair:
    p = ref_pair[0]
    print(f"\nдля сравнения, текущая пара 561.667/545.045: чувств. {p['sens']:.2f}, расхожд. {p['disagree']:.2f},"
          f" Δнаклон {p['dslope']:.2f}")
with open(W / "xe2_ne_pairs.csv", "w", newline="") as fh:
    wr = csv.DictWriter(fh, fieldnames=list(pairs[0])); wr.writeheader(); wr.writerows(sorted(pairs, key=lambda p: -p["sens"]))
print("->", W / "xe2_ne_pairs.csv")

## 7. Таблица линий для общей подгонки

Все линии-кандидаты раздела 6 (видимые, без бленд, не насыщены хотя бы в одном листе A3) с
данными, нужными CRM: ветвление (DBSR; измеренное, где есть), согласие двух форм A (vel/len),
k(1→u) и рычаг метастабилей в обоих наборах, наклон. Подгонка по всем линиям сразу:
неизвестные ne, Te (и плотность ионов как общий множитель); минимизируется сумма
[ln(I_изм/I_CRM)]² по линиям с весами 1/σ². Линии, которые систематически не ложатся, —
признак плохих атомных данных для их верхнего уровня (смотрите `vel/len` и `BR_источник`).

In [ ]:
fit_rows = []
for r in sorted(lines_c, key=lambda r: float(r["wl_nm"])):
    u, lo = int(r["upper_no"]), int(r["lower_no"]) if r["lower_no"] else None
    brd = br.get(u, {}).get(lo) if lo else None                    # ветвление DBSR (раздел 2)
    ratio = meas.get((u, lo)) if lo else None                      # A_истинное / A_наш по спектру
    g = gauge.get((u, lo)) if lo else None
    fit_rows.append(dict(wl_nm=float(r["wl_nm"]), upper=u, upper_label=ref.label(u), lower=lo,
                         lower_label=ref.label(lo) if lo else "", BR_dbsr=brd,
                         BR_best=brd * ratio if (brd and ratio) else brd,
                         BR_source="спектр" if ratio else "DBSR", vel_len=g,
                         A_reliable=(g is not None and 0.7 < g < 1.4),
                         lever_our=lever[u]["LD"], lever_wang=lever[u]["LW"], Te_slope=lever[u]["s"],
                         best_SN=best_sn(r), n_unsaturated_sheets=n_unsat(r)))
with open(W / "xe2_lines_for_fit.csv", "w", newline="") as fh:
    wr = csv.DictWriter(fh, fieldnames=list(fit_rows[0])); wr.writeheader(); wr.writerows(fit_rows)
print(f"{len(fit_rows)} линий ->", W / "xe2_lines_for_fit.csv")
print("из них с надёжной A (0.7 < vel/len < 1.4):", sum(r["A_reliable"] for r in fit_rows))